# Solving a Real Physics PDE: The 1D Poisson Equation

This notebook solves the one-dimensional Poisson equation

$$-\frac{d^2 u}{dx^2} = f(x)$$

on `[0, 1]` with Dirichlet boundary conditions. After finite-difference discretization, this becomes a linear system

$$Lu=f,$$

where `L` is the positive definite discrete negative Laplacian. We solve the system using an inverse-like bounded polynomial transform, which is the kind of spectral primitive QSVT can express after block encoding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsvt.design import design_inverse_polynomial
from qsvt.polynomials import eval_polynomial
from qsvt.pde import dirichlet_laplacian_1d
from qsvt.spectral import apply_polynomial_to_hermitian, eigh_hermitian

np.set_printoptions(precision=4, suppress=True)

## Discretize the boundary-value problem

Use the standard second-difference negative Laplacian with zero boundary values. The source term combines smooth Fourier modes so the exact continuum behavior is easy to interpret.

In [ ]:
# A deliberately small grid keeps the condition number in the range where
# the package's current educational inverse-polynomial helper is effective.
n_points = 6
x, L = dirichlet_laplacian_1d(n_points)

source = np.sin(np.pi * x) + 0.35 * np.sin(3.0 * np.pi * x)
eigenvalues, _ = eigh_hermitian(L)

eigenvalues[0], eigenvalues[-1]

## Normalize the operator for an inverse polynomial

`design_inverse_polynomial(gamma, degree)` approximates `gamma / y` on the positive part of the spectrum. For a positive definite Laplacian, set

$$A = L / \lambda_{\max}, \qquad \gamma = \lambda_{\min}/\lambda_{\max}.$$

Then

$$L^{-1} \approx \frac{P(A)}{\gamma\lambda_{\max}}.$$

The condition number controls how small `gamma` is. Smaller `gamma` generally needs a higher-degree polynomial, and high-resolution PDE grids quickly become challenging for the simple inverse helper used in v0.1.10.

In [ ]:
lambda_min = eigenvalues[0]
lambda_max = eigenvalues[-1]
gamma = lambda_min / lambda_max
condition_number = lambda_max / lambda_min

A = L / lambda_max

gamma, condition_number

## Solve with the inverse-like polynomial

The polynomial solution is compared to NumPy's direct linear solve for the same finite-difference system.

In [ ]:
degree = 45
coeffs = design_inverse_polynomial(gamma=gamma, degree=degree)

inverse_like = apply_polynomial_to_hermitian(A, coeffs) / (gamma * lambda_max)
u_poly = inverse_like @ source

u_direct = np.linalg.solve(L, source)
relative_error = np.linalg.norm(u_poly - u_direct) / np.linalg.norm(u_direct)

relative_error

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(x, source, "o-", label="source f(x)")
axes[0].plot(
    x, u_direct / np.max(np.abs(u_direct)), "o-", label="direct solution, normalized"
)
axes[0].plot(
    x, u_poly / np.max(np.abs(u_poly)), "--", label="polynomial solution, normalized"
)
axes[0].set_xlabel("position")
axes[0].legend()

ys = np.linspace(gamma, 1.0, 600)
axes[1].plot(ys, gamma / ys, label="target gamma / y")
axes[1].plot(ys, eval_polynomial(coeffs, ys), "--", label="inverse-like polynomial")
axes[1].scatter(
    eigenvalues / lambda_max,
    gamma / (eigenvalues / lambda_max),
    color="tab:red",
    s=18,
    zorder=3,
    label="Laplacian spectrum",
)
axes[1].set_xlabel("normalized Laplacian eigenvalue")
axes[1].set_ylabel("normalized inverse weight")
axes[1].legend()

fig.suptitle(
    f"1D Poisson equation via inverse polynomial, relative error={relative_error:.2e}"
)
plt.tight_layout()
plt.show()

## What this demonstrates

The Poisson equation is an elliptic physics PDE, and its discretization is a linear-system problem. This example shows the package's inverse-like QSVT polynomial acting as a regularized spectral inverse for a concrete PDE operator.